In [ ]:
# detect the intention from all PR/issue/commit text fields regardless of strt/end

In [33]:
"""
Flat intention labeling for commit/PR/Issue rows using a weighted keyword dictionary.

Supports TWO dictionary formats (auto-detected):
1) WIDE format:
   - one column per label, one keyword per cell
   - optional per-row weights via <Label>__weight
   - optional Blacklist column (+ optional Blacklist__weight)

2) LONG/TIDY format:
   - columns: label, keyword, optional weight (weight|keyword_weight|kw_weight)
   - if label == "Blacklist" (case-insensitive), goes to blacklist

Hardcoded paths (Windows):
  BASE_DIR        = ...\3 - RQ3_2\Intention_Detection_2
  INPUT_CSV       = All_Commits_PR_Msg_Iss.csv
  DICTIONARY_CSV  = dictionary.csv
  OUTPUT_DIR      = outputs\
  OUTPUT_CSV      = outputs\commits_with_intentions.csv
  KEYWORD_EXT_CSV = outputs\keyword_extension_candidates.csv

Key behavior:
- Labels EVERY row (no start/end boundaries).
- Uses commit + PR + issue text together.
- Issue fields are boosted by ISSUE_BOOST = 1.5.
- Multi-label is NOT treated as weak signal:
    confidence uses separation between the *selected* labels vs the next-best unselected label.
- Generates keyword extension suggestions from UNCLASSIFIED rows.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import pandas as pd

# ---- Try stemming; fallback to no-stem if nltk isn't available ----
try:
    from nltk.stem.snowball import SnowballStemmer
    _stemmer = SnowballStemmer("english")

    def _stem(token: str) -> str:
        return _stemmer.stem(token)

except Exception:
    def _stem(token: str) -> str:
        return token


# =========================
# Hardcoded I/O (as you requested)
# =========================
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection_2")
INPUT_CSV = BASE_DIR / "All_Commits_PR_Msg_Iss.csv"
DICTIONARY_CSV = BASE_DIR / "dictionary.csv"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_CSV = OUTPUT_DIR / "commits_with_intentions.csv"
KEYWORD_EXTENSION_CSV = OUTPUT_DIR / "keyword_extension_candidates.csv"


# =========================
# Your schema (from All_Commits_PR_Msg_Iss.csv)
# =========================
COMMIT_FIELDS = ["commit_subject", "commit_body", "commit_full_message"]
PR_FIELDS     = ["pr_titles", "pr_bodies", "pr_comments_and_reviews"]
ISSUE_FIELDS  = ["issue_titles", "issue_bodies", "issue_comments"]

ISSUE_BOOST = 1.5  # <-- your requested coefficient


# =========================
# Candidate mining controls
# =========================
STOP = set("""
a an the and or but if then else when while of to for in on at by from as is are was were be been being with without
this that these those it its i you we they he she them his her our your their into over under up down out off via vs
add adds added adding remove removed removing fix fixes fixed fixing update updates updated updating change changes changed changing
refactor refactors refactored refactoring bump bumps bumped merge merges merged revert reverts reverted
""".split())

IMPORTANT_SHORT = {"ci", "ui", "e2e", "ftl", "avd", "apk", "aab", "gmd"}

CAND_TOP_N = 200
CAND_MIN_FREQ = 5
CAND_RATIO = 1.5
CAND_MIN_ASSOC = 3
CAND_USE_BIGRAMS = True


# =========================
# Core text / dictionary utils
# =========================
def normalize_text_for_tokens(text: str) -> str:
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)  # de-camelcase
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


def tokenize_and_stem(text: str) -> List[str]:
    """
    Keeps digits (e2e, v2, aab, etc).
    Stems only purely alphabetic tokens.
    """
    if not text:
        return []
    text = normalize_text_for_tokens(text)
    raw = re.findall(r"[a-z0-9]+", text)
    out: List[str] = []
    for t in raw:
        if any(ch.isdigit() for ch in t):
            out.append(t)
        else:
            out.append(_stem(t))
    return out


def count_phrase_occurrences(tokens: List[str], phrase_tokens: List[str]) -> int:
    if not phrase_tokens or not tokens:
        return 0
    if len(phrase_tokens) == 1:
        p = phrase_tokens[0]
        return sum(1 for t in tokens if t == p)
    n = len(phrase_tokens)
    cnt = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            cnt += 1
    return cnt


def _read_dictionary_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dictionary file not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported dictionary format: {path} (use .csv/.xlsx/.xls)")


def _safe_str(v) -> str:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s


# -------------------------
# Dictionary loading: WIDE
# -------------------------
def load_dictionary_wide(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Wide dictionary format:
      - one column per label, one keyword per cell
      - optional per-row weights via <Label>__weight
      - optional Blacklist column (+ optional Blacklist__weight)

    Returns:
      dict_terms: label -> list of {keyword, weight, stem_tokens}
      blacklist_terms: list of {keyword, weight, stem_tokens}
    """
    df = _read_dictionary_table(path)

    # Detect "Blacklist" column name (case-insensitive)
    blacklist_col = None
    for c in df.columns:
        if str(c).strip().lower() == "blacklist":
            blacklist_col = c
            break

    # Label columns = non-weight columns excluding Blacklist and optional "No"
    labels: List[str] = []
    for c in df.columns:
        c_str = str(c)
        if c_str.endswith("__weight"):
            continue
        if blacklist_col is not None and c == blacklist_col:
            continue
        if str(c).strip().lower() == "no":
            continue
        labels.append(c_str)

    if not labels:
        raise ValueError(f"No label columns found in dictionary: {path}")

    def add_items_for_column(series: pd.Series, weights: Optional[pd.Series]) -> List[Dict]:
        items: List[Dict] = []
        for idx, cell in series.items():
            if pd.isna(cell):
                continue
            cell_s = str(cell).strip()
            if not cell_s:
                continue

            for kw in re.split(r"[;,/|]", cell_s):
                kw = kw.strip()
                if not kw:
                    continue

                wt = 1.0
                if weights is not None:
                    wv = weights.iloc[idx]
                    if not pd.isna(wv):
                        try:
                            wt = float(wv)
                        except Exception:
                            wt = 1.0

                stem_tokens = tokenize_and_stem(kw)
                if stem_tokens:
                    items.append({"keyword": kw, "weight": wt, "stem_tokens": stem_tokens})

        # de-dup by stem sequence
        seen = set()
        uniq = []
        for it in items:
            k = " ".join(it["stem_tokens"])
            if k in seen:
                continue
            seen.add(k)
            uniq.append(it)
        return uniq

    dict_terms: Dict[str, List[Dict]] = {}
    for label in labels:
        wcol = f"{label}__weight"
        weights = df[wcol] if wcol in df.columns else None
        dict_terms[label] = add_items_for_column(df[label], weights) if label in df.columns else []

    blacklist_terms: List[Dict] = []
    if blacklist_col is not None:
        bwcol = f"{blacklist_col}__weight"
        bweights = df[bwcol] if bwcol in df.columns else None
        blacklist_terms = add_items_for_column(df[blacklist_col], bweights)

    # remove empty labels (possible if a column exists but has no keywords)
    dict_terms = {k: v for k, v in dict_terms.items() if v}

    if not dict_terms:
        raise ValueError("Wide dictionary loaded, but no keywords found under any label columns.")

    return dict_terms, blacklist_terms


# -------------------------
# Dictionary loading: LONG
# -------------------------
def load_dictionary_long(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Long dictionary format:
      columns like: label, keyword, (optional) weight
      special label value "Blacklist" (case-insensitive) goes to blacklist_terms.
    """
    df = _read_dictionary_table(path)

    # case-insensitive column mapping
    colmap = {str(c).strip().lower(): c for c in df.columns}
    if "label" not in colmap or "keyword" not in colmap:
        raise ValueError("Long dictionary requires columns: label, keyword (case-insensitive).")

    label_col = colmap["label"]
    keyword_col = colmap["keyword"]

    # optional weight column
    weight_col = None
    for cand in ["weight", "keyword_weight", "kw_weight"]:
        if cand in colmap:
            weight_col = colmap[cand]
            break

    dict_terms: Dict[str, List[Dict]] = defaultdict(list)
    blacklist_terms: List[Dict] = []

    for _, row in df.iterrows():
        lab = _safe_str(row.get(label_col, "")).strip()
        kw_cell = _safe_str(row.get(keyword_col, "")).strip()
        if not lab or not kw_cell:
            continue

        wt = 1.0
        if weight_col is not None:
            wv = row.get(weight_col, None)
            if not (wv is None or (isinstance(wv, float) and pd.isna(wv))):
                try:
                    wt = float(wv)
                except Exception:
                    wt = 1.0

        # allow multiple keywords per cell
        for kw in re.split(r"[;,/|]", kw_cell):
            kw = kw.strip()
            if not kw:
                continue
            stem_tokens = tokenize_and_stem(kw)
            if not stem_tokens:
                continue

            item = {"keyword": kw, "weight": wt, "stem_tokens": stem_tokens}

            if lab.lower() == "blacklist":
                blacklist_terms.append(item)
            else:
                dict_terms[lab].append(item)

    # de-dup within each label by stem sequence
    def dedup(items: List[Dict]) -> List[Dict]:
        seen = set()
        out = []
        for it in items:
            k = " ".join(it["stem_tokens"])
            if k in seen:
                continue
            seen.add(k)
            out.append(it)
        return out

    dict_terms = {lab: dedup(items) for lab, items in dict_terms.items()}
    blacklist_terms = dedup(blacklist_terms)

    if not dict_terms:
        raise ValueError("No usable label/keyword pairs found in long dictionary.")

    return dict_terms, blacklist_terms


def load_dictionary_auto(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Auto-detect dictionary format.
    - If columns include (label, keyword) => long format
    - else => wide format
    """
    df = _read_dictionary_table(path)
    cols = {str(c).strip().lower() for c in df.columns}
    if "label" in cols and "keyword" in cols:
        return load_dictionary_long(path)
    return load_dictionary_wide(path)


# =========================
# Build row text parts
# =========================
def build_field_texts(row: pd.Series) -> List[Tuple[str, float]]:
    """
    Returns list of (text, multiplier). Issue fields get ISSUE_BOOST.
    """
    parts: List[Tuple[str, float]] = []

    # Commit
    commit_text = "\n".join([_safe_str(row.get(c, "")) for c in COMMIT_FIELDS]).strip()
    if commit_text:
        parts.append((commit_text, 1.0))

    # PR
    pr_text = "\n".join([_safe_str(row.get(c, "")) for c in PR_FIELDS]).strip()
    if pr_text:
        parts.append((pr_text, 1.0))

    # Issues (boosted)
    issue_text = "\n".join([_safe_str(row.get(c, "")) for c in ISSUE_FIELDS]).strip()
    if issue_text:
        parts.append((issue_text, ISSUE_BOOST))

    return parts


# =========================
# Classification
# =========================
def classify_parts(
    parts: List[Tuple[str, float]],
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    cap_per_keyword_per_part: bool = True,
) -> Tuple[Dict[str, float], Dict[str, List[str]], List[str]]:
    """
    Scores labels by summing over parts. Each part has its multiplier.
    """
    token_parts: List[Tuple[List[str], float]] = []
    for text, mult in parts:
        toks = tokenize_and_stem(text)
        if toks:
            token_parts.append((toks, mult))

    scores = {label: 0.0 for label in dict_terms}
    matches = {label: [] for label in dict_terms}

    # blacklist hits (noise indicators)
    blacklist_hits: List[str] = []
    for it in blacklist_terms:
        for toks, _mult in token_parts:
            if count_phrase_occurrences(toks, it["stem_tokens"]) > 0:
                blacklist_hits.append(it["keyword"])
                break

    # label scoring
    for label, items in dict_terms.items():
        for it in items:
            total_occ = 0.0
            hit = False
            for toks, mult in token_parts:
                occ = count_phrase_occurrences(toks, it["stem_tokens"])
                if occ > 0:
                    hit = True
                    if cap_per_keyword_per_part:
                        occ = 1
                    total_occ += (occ * mult)
            if hit:
                scores[label] += it["weight"] * total_occ
                matches[label].append(it["keyword"])

    return scores, matches, sorted(set(blacklist_hits))


def assign_labels_multi(
    scores: Dict[str, float],
    matches: Dict[str, List[str]],
    blacklist_hits: List[str],
    min_score: float = 1.0,
    multi_ratio: float = 0.8,
    max_labels: int = 3,
) -> Dict[str, object]:
    """
    Multi-label selection:
      choose all labels with score >= top_score * multi_ratio (and >= min_score), up to max_labels.

    Confidence is computed vs the best *unselected* label.
    """
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0] if items else ("", 0.0)
    second_label, second_score = items[1] if len(items) > 1 else ("", 0.0)

    total = sum(scores.values())
    conf_share_top = (top_score / total) if total > 0 else 0.0

    if top_score < min_score:
        label = "UNCLASSIFIED_NOISE" if blacklist_hits else "UNCLASSIFIED"
        return {
            "label_str": label,
            "top_label": top_label, "top_score": float(top_score),
            "second_label": second_label, "second_score": float(second_score),
            "total_score": float(total),
            "selected_score_sum": 0.0,
            "confidence_share_top": float(conf_share_top),
            "confidence_share_selected": 0.0,
            "confidence_margin_selected": 0.0,
            "matched_keywords": "",
            "blacklist_hits": "; ".join(blacklist_hits),
        }

    chosen: List[str] = []
    chosen_scores: List[float] = []
    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
            chosen_scores.append(sc)
        if len(chosen) >= max_labels:
            break

    next_unselected = 0.0
    chosen_set = set(chosen)
    for lab, sc in items:
        if lab not in chosen_set:
            next_unselected = sc
            break

    selected_sum = float(sum(chosen_scores))
    conf_share_selected = (selected_sum / total) if total > 0 else 0.0
    min_selected = float(min(chosen_scores)) if chosen_scores else 0.0
    conf_margin_selected = ((min_selected - next_unselected) / min_selected) if min_selected > 0 else 0.0

    matched = sorted({kw for lab in chosen for kw in matches.get(lab, [])})

    return {
        "label_str": " || ".join(chosen),
        "top_label": top_label, "top_score": float(top_score),
        "second_label": second_label, "second_score": float(second_score),
        "total_score": float(total),
        "selected_score_sum": float(selected_sum),
        "confidence_share_top": float(conf_share_top),
        "confidence_share_selected": float(conf_share_selected),
        "confidence_margin_selected": float(conf_margin_selected),
        "matched_keywords": "; ".join(matched),
        "blacklist_hits": "; ".join(blacklist_hits),
    }


def confidence_bucket(label_str: str, selected_score_sum: float, margin_selected: float, share_selected: float) -> str:
    if label_str.startswith("UNCLASSIFIED"):
        return "UNCLASSIFIED"
    high = (selected_score_sum >= 3.0 and margin_selected >= 0.60 and share_selected >= 0.70)
    med = (
        (selected_score_sum >= 2.0 and margin_selected >= 0.30 and share_selected >= 0.55)
        or (selected_score_sum >= 4.0 and share_selected >= 0.60)
    )
    if high:
        return "HIGH"
    if med:
        return "MEDIUM"
    return "LOW"


# =========================
# Keyword extension (candidate mining)
# =========================
def _valid_token_for_mining(t: str) -> bool:
    if t in STOP:
        return False
    if len(t) >= 3:
        return True
    return t in IMPORTANT_SHORT


def ngrams(tokens: List[str], n: int) -> List[str]:
    if n <= 1:
        return tokens[:]
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def build_keyword_extension_candidates(
    df_with_labels: pd.DataFrame,
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    label_col: str = "label_str",
    top_n: int = 200,
    min_freq: int = 5,
    ratio: float = 1.5,
    min_assoc: int = 3,
    use_bigrams: bool = True,
) -> pd.DataFrame:
    # existing dictionary/blacklist stems (avoid suggesting duplicates)
    dict_stem_seqs = set()
    for items in dict_terms.values():
        for it in items:
            dict_stem_seqs.add(" ".join(it["stem_tokens"]))
    for it in blacklist_terms:
        dict_stem_seqs.add(" ".join(it["stem_tokens"]))

    # count candidates from UNCLASSIFIED rows
    cand_counts = defaultdict(int)
    uncls = df_with_labels[df_with_labels[label_col].astype(str).isin(["UNCLASSIFIED", "UNCLASSIFIED_NOISE"])]

    for _, row in uncls.iterrows():
        parts = build_field_texts(row)
        text = "\n".join([p[0] for p in parts])  # mining uses raw text (no weighting needed)
        toks = [t for t in tokenize_and_stem(text) if _valid_token_for_mining(t)]

        for t in toks:
            if t in dict_stem_seqs:
                continue
            cand_counts[t] += 1

        if use_bigrams and len(toks) >= 2:
            for bg in ngrams(toks, 2):
                if bg in dict_stem_seqs:
                    continue
                cand_counts[bg] += 1

    cands = [(t, c) for t, c in cand_counts.items() if c >= min_freq]
    cands.sort(key=lambda x: x[1], reverse=True)
    cands = cands[:top_n]

    labeled = df_with_labels[~df_with_labels[label_col].astype(str).isin(["UNCLASSIFIED", "UNCLASSIFIED_NOISE"])].copy()
    if labeled.empty or not cands:
        return pd.DataFrame(columns=["keyword", "suggested_labels", "suggested_weight", "freq_unclassified"])

    labeled["primary"] = labeled[label_col].astype(str).map(lambda s: (s.split("||")[0].strip() if s else ""))

    labeled_token_sets = []
    labeled_bigram_sets = []
    for _, row in labeled.iterrows():
        parts = build_field_texts(row)
        text = "\n".join([p[0] for p in parts])
        toks = [t for t in tokenize_and_stem(text) if _valid_token_for_mining(t)]
        labeled_token_sets.append(set(toks))
        labeled_bigram_sets.append(set(ngrams(toks, 2)) if use_bigrams else set())

    labels = list(dict_terms.keys())
    suggestions = []

    for cand, freq in cands:
        counts = {lab: 0 for lab in labels}
        is_bigram = (" " in cand)

        for prim, uni_set, bi_set in zip(labeled["primary"].tolist(), labeled_token_sets, labeled_bigram_sets):
            present = (cand in bi_set) if is_bigram else (cand in uni_set)
            if present and prim in counts:
                counts[prim] += 1

        items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
        top_lab, top_c = items[0]
        top2_lab, top2_c = (items[1] if len(items) > 1 else ("", 0))
        max_other = max([c for lab, c in counts.items() if lab != top_lab], default=0)
        max_rest = max([c for lab, c in counts.items() if lab not in {top_lab, top2_lab}], default=0)

        if top_c >= min_assoc and top_c >= ratio * max_other:
            suggestions.append({
                "keyword": cand,
                "suggested_labels": top_lab,
                "suggested_weight": 2,
                "freq_unclassified": freq,
                **{f"count_{lab}": counts[lab] for lab in labels},
            })
            continue

        if top_c >= min_assoc and top2_c >= min_assoc and top2_c >= ratio * max_rest:
            suggestions.append({
                "keyword": cand,
                "suggested_labels": f"{top_lab} || {top2_lab}",
                "suggested_weight": 1,
                "freq_unclassified": freq,
                **{f"count_{lab}": counts[lab] for lab in labels},
            })

    sug_df = pd.DataFrame(suggestions)
    if not sug_df.empty:
        sug_df.sort_values(["suggested_weight", "freq_unclassified"], ascending=[False, False], inplace=True)
    return sug_df


# =========================
# Main
# =========================
def main() -> None:
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
    if not DICTIONARY_CSV.exists():
        raise FileNotFoundError(f"Dictionary CSV not found: {DICTIONARY_CSV}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(INPUT_CSV)
    dict_terms, blacklist_terms = load_dictionary_auto(DICTIONARY_CSV)

    # Helpful debug: confirm labels loaded are real intention labels
    print("[info] loaded label count:", len(dict_terms))
    print("[info] loaded labels (sample):", list(dict_terms.keys())[:25])

    # Quick schema sanity check (warn but don't crash)
    needed = set(COMMIT_FIELDS + PR_FIELDS + ISSUE_FIELDS)
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print("[warn] Missing expected columns in input CSV:", missing)
        print("       (Code will still run; those fields will just be empty.)")

    results = []
    for _, row in df.iterrows():
        parts = build_field_texts(row)
        scores, matches, black = classify_parts(parts, dict_terms, blacklist_terms)
        res = assign_labels_multi(scores, matches, black, min_score=1.0, multi_ratio=0.8, max_labels=3)
        res["confidence_level"] = confidence_bucket(
            res["label_str"],
            res["selected_score_sum"],
            res["confidence_margin_selected"],
            res["confidence_share_selected"],
        )
        results.append(res)

    out = pd.concat([df, pd.DataFrame(results)], axis=1)
    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print("[ok] wrote:", OUTPUT_CSV)

    # Keyword extension output (next to commits_with_intentions.csv)
    ext = build_keyword_extension_candidates(
        out,
        dict_terms=dict_terms,
        blacklist_terms=blacklist_terms,
        label_col="label_str",
        top_n=CAND_TOP_N,
        min_freq=CAND_MIN_FREQ,
        ratio=CAND_RATIO,
        min_assoc=CAND_MIN_ASSOC,
        use_bigrams=CAND_USE_BIGRAMS,
    )
    ext.to_csv(KEYWORD_EXTENSION_CSV, index=False, encoding="utf-8")
    print("[ok] wrote:", KEYWORD_EXTENSION_CSV)


if __name__ == "__main__":
    main()


[info] loaded label count: 8
[info] loaded labels (sample): ['CI_legacy_provider_config_or_migration', 'CI_github_actions_workflows', 'Add_or_expand_instrumentation_test_suite', 'Wire_and_run_instrumentation_tests_in_CI', 'Toolchain_compatibility_upgrade', 'Switch_execution_backend', 'Stabilize_deflake_instrumentation_in_CI', 'Decommission_disable_cleanup']
[ok] wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection_2\outputs\commits_with_intentions.csv
[ok] wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection_2\outputs\keyword_extension_candidates.csv


In [19]:
# appending the intentions to the episodes list

In [23]:
from pathlib import Path
import pandas as pd

# =========================
# Paths (edit BASE_DIR only)
# =========================
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection_2\outputs")

COMMITS_FILE  = BASE_DIR / "commits_with_intentions.csv"
EPISODES_FILE = BASE_DIR / "All_episodes_with_messages.csv"
OUT_FILE      = BASE_DIR / "List_Change_Episodes_Intention.csv"  # keep your requested name

# =========================
# Load
# =========================
commits = pd.read_csv(COMMITS_FILE)
episodes = pd.read_csv(EPISODES_FILE)

# =========================
# Minimal columns we need from commits
# (keep extra confidence/score columns if they exist)
# =========================
want_cols = ["repo_name", "commit_sha", "top_label", "intent_labels"]
for extra in ["conf_level", "top_score", "conf_share", "conf_margin"]:
    if extra in commits.columns:
        want_cols.append(extra)

commits_small = commits[want_cols].copy()

# Ensure 1 row per (repo_name, commit_sha)
commits_small = commits_small.drop_duplicates(subset=["repo_name", "commit_sha"], keep="last")

# =========================
# Merge START commit intention onto episodes
# =========================
start_map = commits_small.rename(columns={
    "commit_sha": "episode_start_commit_sha",
    "top_label": "start_top_label",
    "intent_labels": "start_intent_labels",
    "conf_level": "start_conf_level",
    "top_score": "start_top_score",
    "conf_share": "start_conf_share",
    "conf_margin": "start_conf_margin",
})

episodes_out = episodes.merge(
    start_map,
    on=["repo_name", "episode_start_commit_sha"],
    how="left",
)

# =========================
# Merge END commit intention onto episodes
# =========================
end_map = commits_small.rename(columns={
    "commit_sha": "episode_end_commit_sha",
    "top_label": "end_top_label",
    "intent_labels": "end_intent_labels",
    "conf_level": "end_conf_level",
    "top_score": "end_top_score",
    "conf_share": "end_conf_share",
    "conf_margin": "end_conf_margin",
})

episodes_out = episodes_out.merge(
    end_map,
    on=["repo_name", "episode_end_commit_sha"],
    how="left",
)

# =========================
# Optional: quick QA counts
# =========================
missing_start = episodes_out["start_top_label"].isna().sum()
missing_end = episodes_out["end_top_label"].isna().sum()
print(f"[info] Episodes: {len(episodes_out)}")
print(f"[info] Missing start intention: {missing_start}")
print(f"[info] Missing end intention:   {missing_end}")

# =========================
# Save
# =========================
episodes_out.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")
print(f"[ok] wrote: {OUT_FILE}")


[info] Episodes: 488
[info] Missing start intention: 7
[info] Missing end intention:   398
[ok] wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection_2\outputs\List_Change_Episodes_Intention.csv
